In [1]:
import torch 
import torch.nn as nn 


In [2]:
class PatchEmbedding(nn.Module):
    def __init__(self,in_channels , embed_dim , patch_size =16 ):
        super().__init__()
        self.conv2d = nn.Conv2d(in_channels  , embed_dim , kernel_size=patch_size , stride=patch_size )
    def forward(self ,x):
        x = self.conv2d(x)
        x=x.flatten(start_dim=2)
        return x.transpose(1,2)
        


In [3]:
x = torch.randn(1, 3, 224, 224)
print(PatchEmbedding(3, 768)(x).shape)  # torch.Size([1, 196, 768])

torch.Size([1, 196, 768])


In [5]:
class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3,
                    num_classes=1000, embed_dim=768, depth=12, num_heads=12,
                    ff_dim=3072, dropout=0.1):
        super().__init__()
        self.patchembedding = PatchEmbedding(in_channels , embed_dim , patch_size)
        cls_init = torch.randn(1 ,1 ,embed_dim) *0.02
        self.cls_token = nn.Parameter(cls_init)
        num_patches = (img_size // patch_size) ** 2
        pos_init = torch.rand(1 , num_patches + 1 , embed_dim) * 0.02
        self.pos_embedding = nn.Parameter(pos_init)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim , nhead=num_heads , dim_feedforward=ff_dim , dropout=dropout, activation="gelu" ,batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer , num_layers=depth)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.out = nn.Linear(embed_dim  , num_classes)

    def forward(self , x):
        z = self.patchembedding(x)
        cls_expd = self.cls_token.expand(z.shape[0] , -1 ,-1)
        z = torch.cat((cls_expd , z) , dim=1)
        z += self.pos_embedding
        z = self.dropout(z)
        z = self.encoder(z)
        z = self.layer_norm(z[: , 0])
        logits = self.out(z)
        return logits       

In [9]:
vit_model = ViT(
                img_size=224, patch_size=16, in_channels=3, num_classes=1000, embed_dim=768,
                depth=12, num_heads=12, ff_dim=3072, dropout=0.1)
batch = torch.randn(4, 3, 224, 224)
logits = vit_model(batch)

In [10]:
logits.shape

torch.Size([4, 1000])

In [12]:
from datasets import load_dataset

pets = load_dataset("timm/oxford-iiit-pet")

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub\datasets--timm--oxford-iiit-pet. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\fi

In [13]:
from transformers import ViTForImageClassification, AutoImageProcessor

model_id = "google/vit-base-patch16-224-in21k"
vit_model = ViTForImageClassification.from_pretrained(model_id , num_labels =37)
vit_processor = AutoImageProcessor.from_pretrained(model_id ,use_fast=True)



c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub\models--google--vit-base-patch16-224-in21k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 2085.18it/s]
[transforme

In [14]:
def vit_collate_fn(batch):
    images = [example["image"] for example in batch]
    labels = [example["label"] for example in batch]
    inputs = vit_processor(images ,return_tensors="pt" , do_convert_rgb=True)
    inputs["labels"] = torch.tensor(labels)
    return inputs

In [15]:
from transformers import Trainer , TrainingArguments
args = TrainingArguments("my_pets_vit" ,per_device_train_batch_size=16 , eval_strategy="epoch" ,num_train_epochs=3, remove_unused_columns=False)
trainer = Trainer(model=vit_model , args=args , data_collator=vit_collate_fn , train_dataset=pets["train"] ,eval_dataset=["test"])
train_output = trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 